# NB00: Stylometric Fingerprint

Establish McCarthy's quantitative signature using proper academic methods.
This notebook BLOCKS all other notebooks. Its output (`data/reference/mccarthy_fingerprint.json`)
is the reference for all downstream comparisons.

Methods:
- Burrows' Delta feature extraction (top 150 MFW, z-scored) via faststylometry
- POS tag distributions (unigram + bigram) via spaCy
- Vocabulary richness curves (Yule's K, Honore's R, MTLD, hapax ratio) via lexicalrichness
- Sentence length full distribution + Hartigan's dip test for bimodality
- Punctuation patterns, syllable distributions via textstat

Output: `data/reference/mccarthy_fingerprint.json`

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

from lib.config import CORPUS_PATH, FINGERPRINT_PATH, MCCARTHY_TARGETS
from lib.mccarthy_metrics import (
    mccarthy_metrics, extract_words, extract_sentences,
    pos_distribution, save_fingerprint, count_syllables
)
from lib.statistical import dip_test, jsd

sns.set_theme(style='whitegrid', font_scale=1.1)
print(f'Corpus path: {CORPUS_PATH}')
print(f'Corpus exists: {CORPUS_PATH.exists()}')

## 1. Load Corpus

In [ ]:
# Load corpus, skip header lines if present
raw = CORPUS_PATH.read_text(encoding='utf-8')

# If the file has a header/title section, skip it
# Adjust skip_lines as needed based on your corpus file
lines = raw.split('\n')
# Find where actual prose starts (skip title, copyright, etc.)
corpus = '\n'.join(lines[55:])  # same skip as corpus_analysis.py

print(f'Total characters: {len(corpus):,}')
print(f'Total lines: {len(lines):,}')
print(f'First 200 chars of prose:')
print(corpus[:200])

## 2. Core McCarthy Metrics

In [ ]:
metrics = mccarthy_metrics(corpus)

print('=== CORE METRICS ===')
for key in ['total_words', 'unique_words', 'monosyllable_pct',
            'avg_syllables_per_word', 'and_frequency_pct', 'and_count',
            'quotation_marks', 'avg_sentence_length', 'median_sentence_length',
            'num_sentences', 'type_token_ratio', 'hapax_count', 'hapax_ratio']:
    val = metrics.get(key, 'N/A')
    if isinstance(val, float):
        print(f'  {key:30s} {val:.4f}')
    else:
        print(f'  {key:30s} {val}')

print('\n=== TOP WORDS ===')
for word, count in metrics['top_words']:
    print(f'  {word:12s} {count:,}')

## 3. Sentence Length Distribution + Bimodality Test

In [ ]:
sent_lens = np.array(metrics['sentence_lengths'])

# Hartigan's dip test for bimodality
dip_stat, dip_pval = dip_test(sent_lens)
print(f'Dip test: statistic={dip_stat:.4f}, p-value={dip_pval:.4f}')
print(f'Bimodal: {"YES" if dip_pval < 0.05 else "NO (or marginal)"}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(sent_lens, bins=50, edgecolor='black', alpha=0.7, color='#c44e52')
axes[0].axvline(np.mean(sent_lens), color='k', linestyle='--', label=f'Mean={np.mean(sent_lens):.1f}')
axes[0].axvline(np.median(sent_lens), color='blue', linestyle='--', label=f'Median={np.median(sent_lens):.0f}')
axes[0].set_xlabel('Sentence Length (words)')
axes[0].set_ylabel('Count')
axes[0].set_title('Sentence Length Distribution')
axes[0].legend()

# Bucketed bar chart
dist = metrics['sentence_length_dist']
axes[1].bar(dist.keys(), dist.values(), color='#4c72b0', edgecolor='black')
axes[1].set_xlabel('Word Count Bucket')
axes[1].set_ylabel('Percentage')
axes[1].set_title('Sentence Length Buckets')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/internals/sentence_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Syllable Distribution

In [ ]:
syl_dist = metrics['syllable_dist']
print('Syllable distribution:')
for k, v in sorted(syl_dist.items()):
    bar = '#' * int(v / 2)
    print(f'  {k} syllable: {v:5.1f}% {bar}')

fig, ax = plt.subplots(figsize=(8, 5))
keys = sorted(syl_dist.keys())
vals = [syl_dist[k] for k in keys]
ax.bar([str(k) for k in keys], vals, color='#55a868', edgecolor='black')
ax.set_xlabel('Syllables per Word')
ax.set_ylabel('Percentage')
ax.set_title('Syllable Distribution (Blood Meridian)')
plt.tight_layout()
plt.show()

## 5. Vocabulary Richness (lexicalrichness)

In [ ]:
from lexicalrichness import LexicalRichness

lr = LexicalRichness(corpus)

richness = {
    'ttr': lr.ttr,
    'rttr': lr.rttr,                # Root TTR
    'cttr': lr.cttr,                # Corrected TTR
    'mtld': lr.mtld(threshold=0.72),
    'hdd': lr.hdd(sample_size=42),  # HD-D (vocd-D approximation)
}

# Yule's K from frequency spectrum
words_lower = [w.lower() for w in extract_words(corpus)]
freq = Counter(words_lower)
N = len(words_lower)
V = len(freq)

# Frequency spectrum: how many words appear exactly i times
spectrum = Counter(freq.values())
sum_i2_vi = sum(i * i * vi for i, vi in spectrum.items())
yulesK = 10_000 * (sum_i2_vi - N) / (N * N)
richness['yules_k'] = yulesK

# Honore's R
V1 = spectrum.get(1, 0)  # hapax legomena
if V1 < V:
    honoreR = 100 * np.log(N) / (1 - V1 / V)
else:
    honoreR = 0.0
richness['honores_r'] = honoreR

richness['hapax_count'] = V1
richness['hapax_ratio'] = V1 / V

print('=== VOCABULARY RICHNESS ===')
for k, v in richness.items():
    print(f'  {k:20s} {v:.4f}')

## 6. POS Tag Distributions (spaCy)

In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm')
# Increase max_length for the full corpus
nlp.max_length = len(corpus) + 1000

pos_dist = pos_distribution(corpus, nlp=nlp)

print('=== POS UNIGRAMS (top 15) ===')
uni_sorted = sorted(pos_dist['unigram'].items(), key=lambda x: -x[1])
for tag, prop in uni_sorted[:15]:
    print(f'  {tag:8s} {prop:.4f}')

print('\n=== POS BIGRAMS (top 15) ===')
bi_sorted = sorted(pos_dist['bigram'].items(), key=lambda x: -x[1])
for (t1, t2), prop in bi_sorted[:15]:
    print(f'  {t1:6s} -> {t2:6s} {prop:.4f}')

# Visualize POS unigrams
fig, ax = plt.subplots(figsize=(10, 5))
tags = [t for t, _ in uni_sorted[:12]]
props = [p for _, p in uni_sorted[:12]]
ax.barh(tags[::-1], props[::-1], color='#8172b2', edgecolor='black')
ax.set_xlabel('Proportion')
ax.set_title('POS Tag Distribution (Blood Meridian)')
plt.tight_layout()
plt.show()

## 7. Burrows' Delta Features (faststylometry)

In [ ]:
from faststylometry import Corpus, load_corpus_from_strings, calculate_burrows_delta

# Split corpus into ~5000-word chunks for Delta calculation
words = extract_words(corpus)
chunk_size = 5000
chunks = []
for i in range(0, len(words), chunk_size):
    chunk_words = words[i:i+chunk_size]
    if len(chunk_words) >= chunk_size // 2:  # skip very short final chunk
        chunks.append(' '.join(chunk_words))

print(f'Split corpus into {len(chunks)} chunks of ~{chunk_size} words each')

# Create corpus for faststylometry
# Each chunk is labeled as McCarthy
authors = ['McCarthy'] * len(chunks)
titles = [f'chunk_{i}' for i in range(len(chunks))]

mccarthy_corpus = load_corpus_from_strings(
    authors=authors,
    titles=titles,
    texts=chunks,
)

# Extract top 150 MFW z-scored features
# These become the reference vector for Delta comparisons in NB02
print('\nTop 20 MFW in Blood Meridian:')
all_text = ' '.join(chunks)
word_freq = Counter(all_text.lower().split())
for word, count in word_freq.most_common(20):
    print(f'  {word:12s} {count:,}')

## 8. Punctuation Patterns

In [ ]:
import textstat

# Punctuation counts per 1000 chars
n_chars = len(corpus)
punct = {
    'period': corpus.count('.'),
    'comma': corpus.count(','),
    'apostrophe': corpus.count("'"),
    'question': corpus.count('?'),
    'hyphen': corpus.count('-'),
    'colon': corpus.count(':'),
    'semicolon': corpus.count(';'),
    'double_quote': corpus.count('"'),
    'exclamation': corpus.count('!'),
}

print('=== PUNCTUATION ===')
for name, count in punct.items():
    per_1k = 1000 * count / n_chars
    print(f'  {name:15s} {count:6,}  ({per_1k:.2f} per 1K chars)')

# Textstat readability
print('\n=== READABILITY (textstat) ===')
print(f'  Flesch Reading Ease:    {textstat.flesch_reading_ease(corpus):.1f}')
print(f'  Flesch-Kincaid Grade:   {textstat.flesch_kincaid_grade(corpus):.1f}')
print(f'  Coleman-Liau Index:     {textstat.coleman_liau_index(corpus):.1f}')
print(f'  Automated Readability:  {textstat.automated_readability_index(corpus):.1f}')
print(f'  Dale-Chall Readability: {textstat.dale_chall_readability_score(corpus):.1f}')

## 9. Compile and Save Fingerprint

In [ ]:
fingerprint = {
    'source': 'Blood Meridian (Cormac McCarthy)',
    'corpus_chars': len(corpus),
    'corpus_words': metrics['total_words'],
    # Core McCarthy metrics
    'monosyllable_pct': metrics['monosyllable_pct'],
    'avg_syllables_per_word': metrics['avg_syllables_per_word'],
    'and_frequency_pct': metrics['and_frequency_pct'],
    'quotation_marks': metrics['quotation_marks'],
    'avg_sentence_length': metrics['avg_sentence_length'],
    'median_sentence_length': metrics['median_sentence_length'],
    'num_sentences': metrics['num_sentences'],
    # Distributions
    'sentence_length_dist': metrics['sentence_length_dist'],
    'syllable_dist': metrics['syllable_dist'],
    # Vocabulary richness
    'type_token_ratio': metrics['type_token_ratio'],
    'hapax_count': richness['hapax_count'],
    'hapax_ratio': richness['hapax_ratio'],
    'yules_k': richness['yules_k'],
    'honores_r': richness['honores_r'],
    'mtld': richness['mtld'],
    'hdd': richness['hdd'],
    # POS distributions
    'pos_unigram': pos_dist['unigram'],
    'pos_bigram': {f'{k[0]}->{k[1]}': v for k, v in pos_dist['bigram'].items()},
    # Punctuation
    'punctuation': punct,
    # Bimodality
    'dip_test_statistic': dip_stat,
    'dip_test_pvalue': dip_pval,
}

save_fingerprint(fingerprint, FINGERPRINT_PATH)
print(f'Fingerprint saved to: {FINGERPRINT_PATH}')
print(f'Keys: {list(fingerprint.keys())}')

## 10. Validation Against Known Baselines

In [ ]:
from lib.mccarthy_metrics import compare_to_baseline

comparison = compare_to_baseline(fingerprint)

print('=== VALIDATION AGAINST FINDINGS.MD BASELINES ===')
all_close = True
for key, comp in comparison.items():
    status = comp['status']
    marker = 'OK' if status == 'CLOSE' else 'WARN'
    if status != 'CLOSE':
        all_close = False
    print(f'  [{marker:4s}] {key:30s} target={comp["target"]:.4f}  actual={comp["actual"]:.4f}  diff={comp["diff"]:+.4f}')

if all_close:
    print('\nAll metrics within 10% of FINDINGS.md baselines.')
else:
    print('\nSome metrics diverge from FINDINGS.md. Check corpus loading (skip lines).')